# Unity 카탈로그로 데이터를 정리, 변환 및 로드합니다.

## 부동산 데모 시나리오
이 데모에서는 암스테르담, 로테르담, 위트레흐트, 헤이그, 에인트호번 등 5개 주요 도시에서 운영되는 가상의 네덜란드 부동산 중개업체인 **PropertyPro NL**의 데이터를 사용합니다.
이 기관은 여러 지역 사무소의 부동산 목록을 Unity 카탈로그로 통합합니다. 원시 데이터에는 일반적인 실제 품질 문제(반복된 가져오기로 인한 중복 레코드 및 선택적 속성 속성의 누락된 값)가 포함되어 있습니다.
| Table | Description |
|---|---|
| `properties` | 500 property listings — includes 30 duplicate records and ~56 null values |
| `agents` | 20 real estate agents across five regions |
| `property_updates` | Staging table: 50 price/status updates + 20 new listings (used for MERGE demo) |

아래 셀을 실행하여 데모 스키마와 테이블을 설정하세요.

In [0]:
from scripts.setup import setup, setup_08

spark.sql("USE CATALOG trainer_demo")
setup_08(spark)

## 프로필 데이터

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/cleanse-transform-load-data-into-unity-catalog.profile-data-summary-statistics.png)

### 🎯 데모: 프로필 데이터 — 요약 통계
** 논점 **
- 데이터를 정리하기 전에 보유하고 있는 데이터가 무엇인지 이해하세요.
- `ANALYZE TABLE`은 쿼리 최적화 프로그램에 제공되는 행 개수, 열 최소/최대, Null 개수 및 고유 개수를 수집합니다.
- `DESCRIBE TABLE EXTENDED` 및 `df.summary()`은 이러한 통계를 다양한 세부 수준으로 노출합니다.

**데모 흐름**
1. `properties` 테이블을 분석하여 통계를 수집합니다.
2. `DESC EXTENDED`을 사용하여 열 수준 통계를 검사합니다.
3. DataFrame API를 사용하여 빠른 수치 요약을 생성합니다.

In [0]:
%sql
-- Switch to the demo schema
USE trainer_demo.demo_08;

-- Step 1: Collect statistics for the full table
ANALYZE TABLE properties COMPUTE STATISTICS;

-- Step 2: Collect column-level statistics for key columns
ANALYZE TABLE properties COMPUTE STATISTICS FOR COLUMNS
    listing_price, bedrooms, bathrooms, area_sqm, year_built;

-- Step 3: Inspect collected statistics for listing_price
DESC EXTENDED properties listing_price;


In [0]:

# Step 4: Use the DataFrame API to profile numeric columns
df = spark.table("trainer_demo.demo_08.properties")

# describe() returns count, mean, std, min, max for numeric columns
df.describe("listing_price", "bedrooms", "bathrooms", "area_sqm", "year_built").show()


## 열 데이터 유형 선택

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/cleanse-transform-load-data-into-unity-catalog.choose-appropriate-column-data-types.png)

### 🎯 데모: 열 데이터 유형 선택
** 논점 **
- 유형 선택은 정밀도, 저장 및 쿼리 정확성에 영향을 미칩니다.
- `listing_price`은 `DOUBLE`(부동 소수점)으로 저장됩니다. 부동 소수점 산술에서는 작은 반올림 오류가 발생할 수 있어 재무 계산에 적합하지 않습니다.
- `DECIMAL(12,2)`은 소수점 이하 최대 10자리, 소수점 이하 2자리의 정확한 값을 저장합니다.
- `year_built`은 `INT`(4바이트)로 저장됩니다. `SMALLINT`(2바이트, 범위 -32 768 ~ 32 767)이면 수년 동안 충분합니다.
- 테이블을 다시 작성하지 않고 값을 변환하려면 `CAST`을 사용하십시오.

**데모 흐름**
1. 현재 스키마를 살펴봅니다. 가격은 `DOUBLE`, 건축 연도는 `INT`을 참고하세요.
2. `DOUBLE` 산술의 정밀도 문제를 보여줍니다.
3. 열을 올바른 유형으로 캐스팅하는 유형화된 뷰를 만듭니다.

In [0]:
%sql
USE trainer_demo.demo_08;

-- Step 1: Inspect the current schema
DESCRIBE TABLE properties;

In [0]:
%sql
-- ✅ Provided: run this to observe floating-point imprecision with DOUBLE
SELECT
  -- Does 0.1 + 0.2 = 0.3 with DOUBLE?
  CAST(0.1 AS DOUBLE) + CAST(0.2 AS DOUBLE) as sum_double,
  CAST(0.1 AS DOUBLE) + CAST(0.2 AS DOUBLE) = CAST(0.3 AS DOUBLE)        AS double_exact,   -- returns FALSE
  CAST(0.1 AS DECIMAL(2,1)) + CAST(0.2 AS DECIMAL(2,1)) as sum_decimal,
  CAST(0.1 AS DECIMAL(2,1)) + CAST(0.2 AS DECIMAL(2,1)) = CAST(0.3 AS DECIMAL(2,1))        AS decimal_exact  -- returns TRUE

In [0]:
%sql
-- Step 3: Create a properly typed view of the properties table
CREATE OR REPLACE VIEW properties_typed AS
SELECT
    property_id,
    address,
    neighborhood,
    city,
    property_type,
    bedrooms,
    CAST(bathrooms    AS DECIMAL(3,1)) AS bathrooms,
    CAST(area_sqm     AS DECIMAL(8,2)) AS area_sqm,
    CAST(listing_price AS DECIMAL(12,2)) AS listing_price,
    listing_date,
    CAST(year_built   AS SMALLINT)     AS year_built,
    status,
    agent_id
FROM properties;

SELECT * FROM properties_typed LIMIT 5;

## 중복 및 null 해결

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/cleanse-transform-load-data-into-unity-catalog.resolve-duplicate-missing-null-values.png)

### 🎯 데모: 중복 및 null 해결
** 논점 **
- `properties` 테이블은 여러 지역 사무소 내보내기에서 로드되어 30개의 중복된 속성 레코드가 생성되었습니다.
- 여러 선택적 속성(`bedrooms`, `bathrooms`, `area_sqm`)에는 정보가 캡처되지 않은 null 값이 포함되어 있습니다.
- 중복을 식별하려면 창 기능과 함께 `GROUP BY / HAVING` 또는 `QUALIFY`을 사용하십시오.
- `QUALIFY`과 함께 `ROW_NUMBER()`을 사용하면 각 속성의 가장 최근에 나열된 버전만 유지됩니다.
- 널을 처리하려면 PySpark에서 `dropna()` / `fillna()`을 사용하거나 SQL에서 `IS NULL` / `COALESCE`을 사용하십시오.
- 중복 행을 표시할 때 `ORDER BY property_id`로 정렬하여 결과를 보기 쉽게 만듭니다.

**데모 흐름**
1. 총 행 수와 개별 property_id를 계산하여 중복 항목을 찾습니다.
2. `QUALIFY`을 사용하여 중복 행을 표시하고, `ORDER BY property_id`로 정렬합니다.
3. `ROW_NUMBER()`을(를) 사용하여 중복을 제거합니다.
4. 선택적 열의 Null 값을 계산하고 처리합니다.

In [0]:
%sql
USE trainer_demo.demo_08;

-- Step 1: Compare total rows vs distinct property_ids
SELECT
    COUNT(*)                    AS total_rows,
    COUNT(DISTINCT property_id) AS unique_properties,
    COUNT(*) - COUNT(DISTINCT property_id) AS duplicate_rows
FROM properties;

In [0]:
%sql
-- Step 2: Show duplicate rows using QUALIFY + window function
SELECT *
FROM properties
QUALIFY COUNT(*) OVER (PARTITION BY property_id) > 1
ORDER BY property_id;

In [0]:
%sql
-- Step 3: Deduplicate — keep the record with the most recent listing_date per property_id
CREATE OR REPLACE TABLE properties_clean AS
SELECT * EXCEPT(rn)
FROM (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY property_id ORDER BY listing_date DESC) AS rn
    FROM properties
)
WHERE rn = 1;

In [0]:
%sql
SELECT COUNT(*) AS clean_row_count FROM properties_clean;

In [0]:
%sql
USE trainer_demo.demo_08;

-- Step 4a: Count null values per optional column
SELECT
    COUNT(*)                           AS total_rows,
    COUNT(*) - COUNT(bedrooms)         AS null_bedrooms,
    COUNT(*) - COUNT(bathrooms)        AS null_bathrooms,
    COUNT(*) - COUNT(area_sqm)         AS null_area_sqm
FROM properties_clean;

In [0]:
%sql
-- Step 4b: Replace nulls — use median bedrooms per property type as a fill value
SELECT
    property_id,
    property_type,
    COALESCE(bedrooms,
        CAST(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY bedrooms)
             OVER (PARTITION BY property_type) AS INT)
    ) AS bedrooms_imputed,
    COALESCE(area_sqm, 0.0) AS area_sqm_filled
FROM properties_clean
LIMIT 20;


In [0]:
from pyspark.sql.functions import col, count, when, median

df = spark.table("trainer_demo.demo_08.properties_clean")

# Step 4c: PySpark — count nulls across all relevant columns
null_counts = df.select([
    count(when(col(c).isNull(), 1)).alias(c)
    for c in ["bedrooms", "bathrooms", "area_sqm"]
])
display(null_counts)

In [0]:
# Step 4d: Drop rows where listing_price is null (critical field)
df_valid = df.dropna(subset=["listing_price"])

# Step 4e: Fill optional nulls with sensible defaults
df_filled = df_valid.fillna({
    "bedrooms":  2,
    "bathrooms": 1.0,
    "area_sqm":  0.0
})
display(df_filled.select("property_id", "property_type", "bedrooms", "bathrooms", "area_sqm").limit(10))

## 필터 및 집계를 사용하여 데이터 변환

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/cleanse-transform-load-data-into-unity-catalog.transform-data-filter-group-aggregate.png)

### 🎯 데모: 필터 및 집계를 사용하여 데이터 변환
** 논점 **
- 필터링은 현재 분석 질문과 관련된 레코드만 선택합니다.
- PySpark에서는 `filter()` / `where()`을 사용합니다. 동작이 동일합니다.
- 다중 조건 필터에는 `&` 및 `|`을 사용합니다. 각 조건을 괄호로 묶습니다.
- `groupBy()` + `agg()`은 카테고리별 요약 통계를 계산합니다.
- `HAVING`이 포함된 SQL `GROUP BY`을(를) 사용하면 집계된 결과를 필터링할 수 있습니다.
- **정렬**은 SQL의 `ORDER BY` 또는 PySpark의 `orderBy()`로 결과를 원하는 기준에 따라 내림차순/오름차순으로 정렬합니다.

**데모 흐름**
1. 가격 임계값 이상의 활성 목록을 필터링합니다.
2. 두 번째 조건을 추가합니다(속성 유형별로 필터링).
3. 동네별로 그룹화하고 가격 통계를 계산합니다.
4. `HAVING`을 사용하여 인구가 많은 동네만 유지하세요.
5. **정렬**을 통해 결과를 가격 또는 listing_count 기준으로 정렬하세요.

In [0]:
%sql
USE trainer_demo.demo_08;

-- Step 1: Filter active listings above €400 000
SELECT property_id, address, city, property_type, listing_price, status
FROM properties_clean
WHERE status = 'Active'
  AND listing_price > 400000
ORDER BY listing_price DESC;

In [0]:
%sql
-- Step 2: Narrow further — active Amsterdam apartments only
SELECT property_id, address, neighborhood, listing_price
FROM properties_clean
WHERE status = 'Active'
  AND city = 'Amsterdam'
  AND property_type = 'Apartment'
ORDER BY neighborhood, listing_price;

In [0]:
%sql
-- Step 3: Aggregate — average, min, max price per neighborhood
SELECT
    city,
    neighborhood,
    COUNT(*)                                AS listing_count,
    ROUND(AVG(listing_price), 0)            AS avg_price,
    MIN(listing_price)                      AS min_price,
    MAX(listing_price)                      AS max_price
FROM properties_clean
WHERE status IN ('Active', 'Pending')
GROUP BY city, neighborhood
ORDER BY avg_price DESC;

In [0]:
%sql
-- Step 4: HAVING — neighborhoods with more than 10 active listings
SELECT
    neighborhood,
    COUNT(*) AS active_listings,
    ROUND(AVG(listing_price), 0) AS avg_price
FROM properties_clean
WHERE status = 'Active'
GROUP BY neighborhood
HAVING COUNT(*) > 10
ORDER BY active_listings DESC;

## 조인 및 집합 연산자를 사용하여 데이터 변환

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/cleanse-transform-load-data-into-unity-catalog.transform-data-join-union-intersect-except.png)

### 🎯 데모: 조인 및 집합 연산자를 사용하여 데이터 변환
** 논점 **
- **INNER JOIN**은 두 테이블 모두에서 일치하는 행만 반환합니다. 이를 사용하면 에이전트 정보로 속성 기록을 강화할 수 있습니다.
- **LEFT ANTI JOIN**은 오른쪽 테이블에 일치 항목이 *없음*인 왼쪽 테이블의 행을 반환합니다. 판매된 적이 없는 부동산을 찾는 데 유용합니다.
- **UNION ALL**은 결과 세트를 수직으로 스택합니다. 이를 사용하여 다양한 상태 카테고리의 목록을 결합합니다.
- **INTERSECT**는 두 결과 세트 모두에 있는 행만 반환합니다.
- **정렬**은 SQL의 `ORDER BY` 또는 PySpark의 `orderBy()`로 결과를 원하는 기준에 따라 내림차순/오름차순으로 정렬합니다.

**데모 흐름**
1. 에이전트에 대한 INNER JOIN 속성 — 에이전시 및 에이전트 이름으로 목록을 강화합니다.
2. LEFT ANTI JOIN — `Sold` 레코드가 없는 속성을 찾습니다.
3. UNION ALL — 활성 목록과 판매 목록을 하나의 통합 데이터 세트로 결합합니다.
4. 정렬 — 결과를 listing_price 또는 원하는 기준으로 정렬합니다.

In [0]:
%sql
USE trainer_demo.demo_08;

-- Step 1: INNER JOIN — enrich active listings with agent and agency details
SELECT
    p.property_id,
    p.address,
    p.city,
    p.property_type,
    p.listing_price,
    p.status,
    a.agent_name,
    a.agency
FROM properties_clean  p
INNER JOIN agents       a ON p.agent_id = a.agent_id
WHERE p.status = 'Active'
ORDER BY p.listing_price DESC
LIMIT 20;

In [0]:
%sql
-- Step 2: LEFT ANTI JOIN — properties with no Sold counterpart (never sold)
SELECT p.property_id, p.address, p.city, p.listing_price, p.listing_date
FROM properties_clean p
LEFT ANTI JOIN (
    SELECT DISTINCT property_id FROM properties_clean WHERE status = 'Sold'
) sold ON p.property_id = sold.property_id;

In [0]:
%sql
-- Step 3: UNION ALL — combine active and sold listings with a source label
SELECT property_id, address, city, listing_price, 'Active Listing' AS listing_category
FROM properties_clean
WHERE status = 'Active'
UNION ALL
SELECT property_id, address, city, listing_price, 'Completed Sale' AS listing_category
FROM properties_clean
WHERE status = 'Sold'
ORDER BY listing_price DESC
LIMIT 30;

## 비정규화 및 피벗을 사용하여 데이터 변환

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/cleanse-transform-load-data-into-unity-catalog.transform-data-denormalize-pivot-unpivot.png)

### 🎯 데모: 비정규화 및 피벗을 사용하여 데이터 변환
** 논점 **
- **비정규화**는 정규화된 여러 테이블을 하나의 넓은 테이블로 평면화하여 런타임 조인을 제거하고 BI 쿼리 속도를 높입니다. 이것을 금층에 적용하십시오.
- **PIVOT**은 열(예: 속성 유형)의 고유 값을 별도의 열로 회전하므로 교차표 보고서에 이상적입니다.
- **UNPIVOT**은 이를 반대로 수행합니다. 넓은 열을 다시 행으로 변환합니다. 들어오는 데이터가 넓은 형식으로 도착하지만 파이프라인에서 긴 형식을 기대하는 경우에 유용합니다.
- **정렬**은 SQL의 `ORDER BY` 또는 PySpark의 `orderBy()`로 결과를 원하는 기준에 따라 내림차순/오름차순으로 정렬합니다.

**데모 흐름**
1. 속성과 에이전트를 결합하는 비정규화된 골드 레이어 뷰를 만듭니다.
2. PIVOT 부동산은 이웃 전체에 걸쳐 `property_type`별로 계산됩니다.
3. UNPIVOT은 분기별로 목표를 다시 행으로 나열합니다.
4. 결과를 원하는 기준(예: 가격, listing_count 등)으로 정렬합니다.

In [0]:
%sql
USE trainer_demo.demo_08;

-- Step 1: Denormalization — create a gold-layer view combining properties + agents
CREATE OR REPLACE VIEW gold_property_listings AS
SELECT
    p.property_id,
    p.address,
    p.neighborhood,
    p.city,
    p.property_type,
    p.bedrooms,
    p.area_sqm,
    p.listing_price,
    p.listing_date,
    p.year_built,
    p.status,
    a.agent_name,
    a.agency,
    a.region            AS agent_region
FROM properties_clean p
INNER JOIN agents      a ON p.agent_id = a.agent_id;

SELECT * FROM gold_property_listings LIMIT 10;

In [0]:
%sql
-- Step 2: PIVOT — property count by property_type across neighborhoods (top 6 neighborhoods)
SELECT neighborhood, Apartment, House, Condo, Townhouse, Villa
FROM (
    SELECT neighborhood, property_type
    FROM properties_clean
    WHERE neighborhood IN ('Jordaan', 'De Pijp', 'Kralingen', 'Binnenstad', 'Scheveningen', 'Strijp-S')
)
PIVOT (
    COUNT(*)
    FOR property_type IN (
        'Apartment' AS Apartment,
        'House'     AS House,
        'Condo'     AS Condo,
        'Townhouse' AS Townhouse,
        'Villa'     AS Villa
    )
);

In [0]:
%sql
-- Step 3: UNPIVOT — convert wide quarterly targets back into rows
CREATE OR REPLACE TEMP VIEW neighborhood_targets (neighborhood, q1_target, q2_target, q3_target, q4_target) AS
VALUES
    ('Jordaan',       45, 52, 48, 55),
    ('De Pijp',       38, 41, 39, 44),
    ('Kralingen',     30, 35, 32, 38),
    ('Binnenstad',    25, 28, 27, 31),
    ('Scheveningen',  20, 22, 21, 25),
    ('Strijp-S',      18, 20, 19, 22);

SELECT neighborhood, quarter, target_listings
FROM neighborhood_targets
UNPIVOT (
    target_listings FOR quarter IN (q1_target, q2_target, q3_target, q4_target)
)
ORDER BY neighborhood, quarter;


## 병합, 삽입, 추가를 통해 데이터 로드

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/cleanse-transform-load-data-into-unity-catalog.load-data-merge-insert-append.png)

### 🎯 데모: 병합, 삽입, 추가, 정렬을 통해 데이터 로드
** 논점 **
- **INSERT INTO**는 행을 추가합니다. 겹치지 않는 데이터(새 일일 목록)의 증분 로드에 사용됩니다. 기존 레코드를 확인하지 않습니다.
- **INSERT OVERWRITE**는 전체 테이블 또는 특정 파티션을 대체합니다. 집계된 요약 테이블을 새로 고치는 데 사용됩니다.
- **MERGE**는 upsert 패턴입니다. 일치하는 행을 업데이트하고 새 행을 삽입합니다. 변경 데이터를 동기화하는 데 필수적입니다.
- **정렬**은 SQL의 `ORDER BY` 또는 PySpark의 `orderBy()`로 결과를 원하는 기준에 따라 내림차순/오름차순으로 정렬합니다.
- `WHEN MATCHED` 및 `WHEN NOT MATCHED` 절에는 세부적인 제어를 위한 추가 조건이 있을 수 있습니다.

**데모 흐름**
1. INSERT INTO — 세 개의 새로운 속성 목록을 추가합니다.
2. 덮어쓰기 삽입 - 주변 가격 요약을 새로 고칩니다.
3. 병합 — `property_updates` 준비 테이블을 `properties_clean`에 적용합니다.
4. 정렬 — 결과를 가격 등 원하는 기준으로 정렬합니다.

In [0]:
%sql
USE trainer_demo.demo_08;

-- ── Step 1: INSERT INTO — append three new listings ──────────────────────────
INSERT INTO properties_clean
    (property_id, address, neighborhood, city, property_type,
     bedrooms, bathrooms, area_sqm, listing_price, listing_date, year_built, status, agent_id)
VALUES
    (501, 'Herengracht 88',    'Centrum',    'Amsterdam', 'Condo',     2, 1.5, 85.0,  325000.0, '2025-03-01', 1998, 'Active', 1),
    (502, 'Coolsingel 14',     'Centrum',    'Rotterdam', 'Apartment', 1, 1.0, 58.0,  189000.0, '2025-03-05', 2010, 'Active', 3),
    (503, 'Stratumseind 77',   'Centrum',    'Eindhoven', 'House',     4, 2.0, 162.0, 498000.0, '2025-03-10', 2003, 'Active', 9);

SELECT COUNT(*) AS total_after_insert FROM properties_clean;

In [0]:
%sql
-- ── Step 2: INSERT OVERWRITE — refresh the neighborhood price summary ─────────
CREATE TABLE IF NOT EXISTS neighborhood_price_summary (
    city            STRING,
    neighborhood    STRING,
    listing_count   BIGINT,
    avg_price       DOUBLE,
    median_price    DOUBLE
);

INSERT OVERWRITE neighborhood_price_summary
SELECT
    city,
    neighborhood,
    COUNT(*)                                            AS listing_count,
    ROUND(AVG(listing_price), 0)                        AS avg_price,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP
          (ORDER BY listing_price), 0)                  AS median_price
FROM properties_clean
WHERE status IN ('Active', 'Pending')
GROUP BY city, neighborhood;

SELECT * FROM neighborhood_price_summary ORDER BY avg_price DESC LIMIT 10;

In [0]:
%sql
-- ── Step 3: MERGE — apply status and price updates from staging table ─────────
MERGE INTO properties_clean AS target
USING property_updates     AS source
ON target.property_id = source.property_id
WHEN MATCHED THEN
    UPDATE SET
        target.status        = source.new_status,
        target.listing_price = source.new_price
WHEN NOT MATCHED THEN
    INSERT (property_id, status, listing_price, listing_date,
            address, neighborhood, city, property_type,
            bedrooms, bathrooms, area_sqm, year_built, agent_id)
    VALUES (source.property_id, source.new_status, source.new_price, source.update_date,
            'Unknown', 'Unknown', 'Unknown', 'Unknown',
            NULL, NULL, NULL, NULL, NULL);

-- Verify: check a sample of updated records
SELECT p.property_id, p.status, p.listing_price, u.new_status, u.new_price
FROM properties_clean  p
JOIN property_updates  u ON p.property_id = u.property_id
LIMIT 10;
